# Rejection option on a toy 2D classification problem

This notebook shows a small synthetic example where a model with an abstention / rejection option
can avoid many mistakes near the decision boundary.

We will:
1. create a 2D, 3-class dataset,
2. train a plain MLP with cross-entropy,
3. train the same MLP with a rejection-option loss,
4. compare the usual test error with the error on the subset of samples the model chooses to accept.


In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_num_threads(1)

DEVICE = torch.device('cpu')
print('Using device:', DEVICE)


## 1) Make a small 2D dataset

Three Gaussian blobs work well here because the class boundaries overlap in a way that is easy to visualize.
The overlap is intentional: it creates a thin ambiguous region where an abstention mechanism can help.


In [ ]:
# Three overlapping classes in 2D
X, y = make_blobs(
    n_samples=3000,
    centers=[(-2, 0), (2, 0), (0, 2.5)],
    cluster_std=[1.2, 1.2, 1.0],
    random_state=seed,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=seed
)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
y_test_t = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

print('Train size:', len(X_train), 'Test size:', len(X_test))
print('Class balance (train):', np.bincount(y_train))

fig, ax = plt.subplots()
for cls in np.unique(y_train):
    idx = y_train == cls
    ax.scatter(X_train[idx, 0], X_train[idx, 1], s=14, alpha=0.55, label=f'class {cls}')
ax.set_title('Training data')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.legend(loc='best')
plt.show()


## 2) Define the model and training utilities

The two models share the exact same MLP backbone. The only difference is the output layer:

- **Cross-entropy model:** 3 outputs, one per class.
- **Rejection model:** 4 outputs, 3 class probabilities plus 1 reservation probability.

For the rejection model, the loss is the one shown in your screenshot, written in a numerically stable way:

\[
L = -
rac{1}{N} \sum_i \log\left(p_{y_i} + 
rac{p_{r}}{	ext{reward}}
ight)
\]

where \(p_{y_i}\) is the probability of the true class and \(p_r\) is the reservation probability.


In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def gambler_loss(model_output: torch.Tensor, targets: torch.Tensor, reward: float = 2.0) -> torch.Tensor:
    '''Rejection-option / gambler loss.

    The last logit is treated as the reservation output.
    '''
    probs = torch.softmax(model_output, dim=1)
    class_probs = probs[:, :-1]
    reservation = probs[:, -1]
    gain = torch.gather(class_probs, dim=1, index=targets.unsqueeze(1)).squeeze(1)
    stable = gain + reservation / reward + 1e-8
    return -torch.log(stable).mean()


def train_model(model, loss_fn, X_train, y_train, epochs=150, lr=5e-3):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        opt.zero_grad()
        logits = model(X_train)
        loss = loss_fn(logits, y_train)
        loss.backward()
        opt.step()
    return model


def evaluate_ce(model):
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax(dim=1)
        acc = (pred == y_test_t).float().mean().item()
        err = 1.0 - acc
        return {
            'accuracy': acc,
            'error': err,
            'pred': pred.cpu().numpy(),
            'probs': probs.cpu().numpy(),
        }


def evaluate_rejection(model, threshold=None):
    with torch.no_grad():
        logits = model(X_test_t)
        probs = torch.softmax(logits, dim=1)
        class_probs = probs[:, :-1]
        reservation = probs[:, -1]
        pred = class_probs.argmax(dim=1)

        if threshold is None:
            accepted = reservation < class_probs.max(dim=1).values
        else:
            accepted = reservation < threshold

        coverage = accepted.float().mean().item()
        accepted_count = max(accepted.float().sum().item(), 1.0)
        accepted_acc = ((pred == y_test_t) & accepted).float().sum().item() / accepted_count
        accepted_err = 1.0 - accepted_acc
        overall_acc = (pred == y_test_t).float().mean().item()

        return {
            'overall_accuracy': overall_acc,
            'coverage': coverage,
            'accepted_accuracy': accepted_acc,
            'accepted_error': accepted_err,
            'pred': pred.cpu().numpy(),
            'accepted': accepted.cpu().numpy(),
            'reservation': reservation.cpu().numpy(),
            'y_true': y_test_t.cpu().numpy(),
            'probs': probs.cpu().numpy(),
        }


## 3) Train the plain cross-entropy model

This model must always pick one of the three classes, so any ambiguous point gets forced into a decision.


In [ ]:
ce_model = SmallMLP(out_dim=3)
ce_model = train_model(
    ce_model,
    loss_fn=lambda logits, targets: F.cross_entropy(logits, targets),
    X_train=X_train_t,
    y_train=y_train_t,
    epochs=120,
    lr=5e-3,
)

ce_metrics = evaluate_ce(ce_model)
print(f"Cross-entropy test accuracy: {ce_metrics['accuracy']:.3f}")
print(f"Cross-entropy test error:    {ce_metrics['error']:.3f}")

fig, ax = plt.subplots()

x_min, x_max = X_train[:, 0].min() - 1.0, X_train[:, 0].max() + 1.0
y_min, y_max = X_train[:, 1].min() - 1.0, X_train[:, 1].max() + 1.0
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 350), np.linspace(y_min, y_max, 350))
grid = np.c_[xx.ravel(), yy.ravel()]
with torch.no_grad():
    grid_t = torch.tensor(grid, dtype=torch.float32, device=DEVICE)
    pred = torch.softmax(ce_model(grid_t), dim=1).argmax(dim=1).cpu().numpy()
ax.contourf(xx, yy, pred.reshape(xx.shape), levels=[-0.5, 0.5, 1.5, 2.5], alpha=0.30)

for cls in np.unique(y_test):
    idx = y_test == cls
    ax.scatter(X_test[idx, 0], X_test[idx, 1], s=12, label=f'class {cls}', edgecolor='none')

mistakes = ce_metrics['pred'] != y_test
ax.scatter(X_test[mistakes, 0], X_test[mistakes, 1], s=40, facecolors='none', edgecolors='k', linewidths=1.3, label='misclassified')
ax.set_title('Plain MLP trained with cross-entropy')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.legend(loc='best', fontsize=9)
plt.show()


## 4) Train the same MLP with the rejection-option loss

The model now has a fourth output that represents the reservation probability.
At inference time, we can reject a sample when the reservation probability is large enough.


In [ ]:
reward = 2.0
rej_model = SmallMLP(out_dim=4)
rej_model = train_model(
    rej_model,
    loss_fn=lambda logits, targets: gambler_loss(logits, targets, reward=reward),
    X_train=X_train_t,
    y_train=y_train_t,
    epochs=150,
    lr=5e-3,
)

rej_metrics_default = evaluate_rejection(rej_model, threshold=None)
print(f"Rejection model overall accuracy: {rej_metrics_default['overall_accuracy']:.3f}")
print(f"Rejection model coverage:        {rej_metrics_default['coverage']:.3f}")
print(f"Accuracy on accepted samples:    {rej_metrics_default['accepted_accuracy']:.3f}")
print(f"Error on accepted samples:       {rej_metrics_default['accepted_error']:.3f}")

fig, ax = plt.subplots()

x_min, x_max = X_train[:, 0].min() - 1.0, X_train[:, 0].max() + 1.0
y_min, y_max = X_train[:, 1].min() - 1.0, X_train[:, 1].max() + 1.0
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 350), np.linspace(y_min, y_max, 350))
grid = np.c_[xx.ravel(), yy.ravel()]
with torch.no_grad():
    grid_t = torch.tensor(grid, dtype=torch.float32, device=DEVICE)
    probs = torch.softmax(rej_model(grid_t), dim=1)
    class_probs = probs[:, :-1]
    reservation = probs[:, -1]
    labels = class_probs.argmax(dim=1).cpu().numpy()
    abstain = (reservation >= class_probs.max(dim=1).values).cpu().numpy()

ax.contourf(xx, yy, labels.reshape(xx.shape), levels=[-0.5, 0.5, 1.5, 2.5], alpha=0.30)
ax.contourf(xx, yy, abstain.reshape(xx.shape), levels=[-0.5, 0.5, 1.5], colors=['none', 'lightgray'], alpha=0.45)

accepted = rej_metrics_default['accepted']
for cls in np.unique(y_test):
    idx = (y_test == cls) & accepted
    ax.scatter(X_test[idx, 0], X_test[idx, 1], s=14, label=f'class {cls} (accepted)', edgecolor='none')
    rej_idx = (y_test == cls) & (~accepted)
    ax.scatter(X_test[rej_idx, 0], X_test[rej_idx, 1], s=28, marker='x', label=f'class {cls} (rejected)')

ax.set_title(f'Rejection model (reward={reward})')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.legend(loc='best', fontsize=8, ncol=2)
plt.show()


## 5) Sweep the abstention threshold

A nice way to summarize the benefit of rejection is to plot the **error on accepted samples** as a function of **coverage**.
As we make the model more willing to abstain, coverage drops, but the error on the retained points usually drops too.


In [ ]:
thresholds = np.linspace(0.00, 0.50, 51)
coverages = []
accepted_errors = []
accepted_accs = []
for th in thresholds:
    m = evaluate_rejection(rej_model, threshold=float(th))
    coverages.append(m['coverage'])
    accepted_errors.append(m['accepted_error'])
    accepted_accs.append(m['accepted_accuracy'])

baseline_error = ce_metrics['error']

fig, ax = plt.subplots()
ax.plot(coverages, accepted_errors, marker='o', markersize=3, linewidth=1.5, label='rejection model')
ax.axhline(baseline_error, linestyle='--', label='plain CE error (no abstention)')
ax.set_xlabel('coverage (fraction accepted)')
ax.set_ylabel('error on accepted samples')
ax.set_title('Selective risk curve')
ax.legend(loc='best')
plt.show()

for target_cov in [0.90, 0.80, 0.70, 0.60]:
    idx = int(np.argmin(np.abs(np.array(coverages) - target_cov)))
    print(
        f"target coverage ~{target_cov:.2f} | actual coverage={coverages[idx]:.3f} | "
        f"accepted error={accepted_errors[idx]:.3f}"
    )


## What to look for

- The plain cross-entropy model must always predict a class, so points near the overlap region are often forced into a mistake.
- The rejection-option model learns to place more probability mass on the reservation output near ambiguous samples.
- Once you allow abstention, the error on the accepted subset falls noticeably.

A useful next step is to tune `reward` and the abstention threshold to get the coverage level you want.
